[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/garrygu/newegg-ai-workshop/blob/main/lv1-beginner-v2/archive/Session_4_Chatbot_Sentiment_Voice.ipynb)


# 💬 Session 4 — Chatbot, Sentiment & Voice

Build AI that understands emotions AND speaks! 🎤

In this session, you'll:
1. 😊 Build a sentiment-aware chatbot
2. 🎤 Add voice input (Speech-to-Text) - *optional*
3. 🔊 Add voice output (Text-to-Speech) - *optional*
4. 🤖 Create a talking game host for Session 5!

---

## 🎯 What You'll Build

```
📝 User Input    →    🧠 Sentiment AI    →    💬 Smart Response
  "I'm excited!"       (detects emotion)       "😊 Great to hear!"
       ↓                                              ↓
🎤 Voice Input                                  🔊 Voice Output
(optional)                                      (optional)
```

---

## ⚙️ Part 1: Setup

In [ ]:
!pip install --quiet transformers torch

In [ ]:
from transformers import pipeline
import random

print("✅ Libraries loaded!")

---

## 😊 Part 2: Sentiment Analysis

**Sentiment Analysis** = Detecting emotions in text

In [ ]:
# Load sentiment analysis model
print("Loading sentiment model...")
sentiment_analyzer = pipeline("sentiment-analysis")
print("✅ Ready!")

In [ ]:
# Test it!
test_texts = [
    "I love this AI workshop!",
    "This is boring and confusing.",
    "The weather is nice today.",
    "I'm so excited to build my game!"
]

print("😊 Sentiment Analysis Results:\n")
for text in test_texts:
    result = sentiment_analyzer(text)[0]
    emoji = "😊" if result['label'] == 'POSITIVE' else "😢"
    print(f"{emoji} \"{text}\"")
    print(f"   → {result['label']} ({result['score']:.1%} confident)\n")

### 🎯 How Sentiment Analysis Works

| Component | Description |
|:--|:--|
| **Input** | Text message |
| **Model** | Transformer (like mini-GPT) |
| **Output** | POSITIVE, NEGATIVE + confidence |
| **Training** | Learned from millions of labeled texts |

---

## 🤖 Part 3: Sentiment-Aware Chatbot

In [ ]:
class SentimentChatbot:
    """A chatbot that responds based on user sentiment."""
    
    def __init__(self, name="Buddy"):
        self.name = name
        self.analyzer = sentiment_analyzer
        
        # Response templates
        self.positive_responses = [
            "😊 That's wonderful to hear! {follow_up}",
            "🎉 Your enthusiasm is contagious! {follow_up}",
            "✨ I love your positive energy! {follow_up}",
            "🌟 Awesome! {follow_up}"
        ]
        
        self.negative_responses = [
            "😔 I'm sorry to hear that. {follow_up}",
            "🤗 That sounds tough. {follow_up}",
            "💙 I understand. {follow_up}",
            "🌈 Don't worry, things will get better! {follow_up}"
        ]
        
        self.neutral_responses = [
            "🤔 Interesting! {follow_up}",
            "📝 I see. {follow_up}",
            "💭 Tell me more! {follow_up}"
        ]
        
        self.follow_ups = [
            "What else is on your mind?",
            "Would you like to continue?",
            "How can I help?"
        ]
    
    def respond(self, user_message):
        """Generate a response based on sentiment."""
        # Analyze sentiment
        result = self.analyzer(user_message)[0]
        sentiment = result['label']
        confidence = result['score']
        
        # Pick response template
        if sentiment == 'POSITIVE' and confidence > 0.6:
            template = random.choice(self.positive_responses)
        elif sentiment == 'NEGATIVE' and confidence > 0.6:
            template = random.choice(self.negative_responses)
        else:
            template = random.choice(self.neutral_responses)
        
        follow_up = random.choice(self.follow_ups)
        return template.format(follow_up=follow_up)
    
    def chat(self):
        """Interactive chat loop."""
        print(f"\n🤖 {self.name}: Hi! I'm {self.name}, your AI buddy!")
        print(f"   Type 'quit' to exit.\n")
        
        while True:
            user_input = input("You: ").strip()
            if user_input.lower() in ['quit', 'exit', 'bye']:
                print(f"\n🤖 {self.name}: Goodbye! 👋")
                break
            
            response = self.respond(user_input)
            print(f"\n🤖 {self.name}: {response}\n")

In [ ]:
# Create and test chatbot
bot = SentimentChatbot("GameBot")

# Test responses
test_messages = [
    "I'm having the best day ever!",
    "I failed my exam and feel terrible.",
    "The sky is blue."
]

print("🤖 Testing GameBot responses:\n")
for msg in test_messages:
    print(f"You: {msg}")
    print(f"GameBot: {bot.respond(msg)}\n")

In [ ]:
# Uncomment to chat interactively!
# bot.chat()

---

## 🎤 Part 4: Voice Input (Optional)

**Speech-to-Text** using OpenAI's Whisper model

> ⏱️ *Skip this section if you're short on time*

In [ ]:
# Install Whisper (optional)
# !pip install --quiet openai-whisper sounddevice scipy

In [ ]:
# Voice input setup (optional)
VOICE_INPUT_ENABLED = False  # Set to True if you installed whisper

if VOICE_INPUT_ENABLED:
    import whisper
    import sounddevice as sd
    from scipy.io.wavfile import write
    
    # Load Whisper model
    print("🎤 Loading Whisper model...")
    whisper_model = whisper.load_model("base")
    print("✅ Voice input ready!")
    
    def record_audio(duration=5, filename="temp_audio.wav"):
        """Record audio from microphone."""
        print(f"🎙️ Recording for {duration} seconds...")
        audio = sd.rec(int(duration * 16000), samplerate=16000, channels=1)
        sd.wait()
        write(filename, 16000, audio)
        print("✅ Recording complete!")
        return filename
    
    def transcribe_audio(filename):
        """Convert audio to text using Whisper."""
        result = whisper_model.transcribe(filename)
        return result["text"]
else:
    print("💡 Voice input disabled. Enable by installing whisper.")

In [ ]:
# Test voice input (if enabled)
if VOICE_INPUT_ENABLED:
    audio_file = record_audio(duration=3)
    text = transcribe_audio(audio_file)
    print(f"🎤 You said: {text}")
    print(f"🤖 Bot: {bot.respond(text)}")

---

## 🔊 Part 5: Voice Output (Optional)

**Text-to-Speech** - Make your chatbot speak!

> ⏱️ *Skip this section if you're short on time*

In [ ]:
# Simple TTS using pyttsx3 (offline, works everywhere)
# !pip install --quiet pyttsx3

In [ ]:
# Voice output setup (optional)
VOICE_OUTPUT_ENABLED = False  # Set to True if you installed pyttsx3

if VOICE_OUTPUT_ENABLED:
    import pyttsx3
    
    engine = pyttsx3.init()
    engine.setProperty('rate', 175)  # Speed
    
    def speak(text):
        """Convert text to speech."""
        # Remove emojis for cleaner speech
        clean_text = ''.join(c for c in text if ord(c) < 0x1F600 or ord(c) > 0x1F64F)
        clean_text = ''.join(c for c in clean_text if ord(c) < 0x1F300 or ord(c) > 0x1F5FF)
        engine.say(clean_text)
        engine.runAndWait()
    
    print("🔊 Voice output ready!")
else:
    def speak(text):
        print(f"🔊 [Would say]: {text}")
    print("💡 Voice output disabled. Enable by installing pyttsx3.")

In [ ]:
# Test voice output
speak("Hello! I am your AI assistant. How are you today?")

---

## 🎮 Part 6: Complete Talking Chatbot

In [ ]:
class TalkingChatbot(SentimentChatbot):
    """A chatbot that can speak its responses."""
    
    def __init__(self, name="GameHost", use_voice=False):
        super().__init__(name)
        self.use_voice = use_voice
    
    def respond(self, user_message, speak_response=True):
        """Generate and optionally speak response."""
        response = super().respond(user_message)
        
        if self.use_voice and speak_response:
            speak(response)
        
        return response
    
    def greet(self):
        """Speak a greeting."""
        greeting = f"Welcome to the AI Game! I'm {self.name}, your host today!"
        print(f"🤖 {self.name}: {greeting}")
        if self.use_voice:
            speak(greeting)

In [ ]:
# Create talking chatbot
host = TalkingChatbot("GameHost", use_voice=VOICE_OUTPUT_ENABLED)
host.greet()

# Test
response = host.respond("I'm really excited to play!")
print(f"\nResponse: {response}")

---

## 🏆 Challenge Zone

### Challenge 1: Add More Emotions

Add support for more emotions like ANGRY, EXCITED, SAD!

In [ ]:
# Load emotion classifier (detects 6 emotions!)
try:
    emotion_classifier = pipeline(
        "text-classification",
        model="j-hartmann/emotion-english-distilroberta-base",
        top_k=None
    )
    
    # Test it
    result = emotion_classifier("I'm so angry about this!")
    print("Emotions detected:")
    for emotion in result[0][:3]:  # Top 3
        print(f"  {emotion['label']}: {emotion['score']:.1%}")
except:
    print("⚠️ Emotion model not available. Using basic sentiment.")

### Challenge 2: Custom Responses

Add your own creative response templates!

In [ ]:
# Add these to your chatbot!
my_custom_responses = {
    "positive": [
        "🚀 [YOUR CUSTOM POSITIVE RESPONSE]",
    ],
    "negative": [
        "💪 [YOUR CUSTOM SUPPORTIVE RESPONSE]",
    ]
}

---

## 💾 Save for Session 5

In [ ]:
# Save the chatbot class for use in Session 5
import os
os.makedirs("game_modules", exist_ok=True)

chatbot_code = '''
from transformers import pipeline
import random

class GameChatbot:
    """Sentiment-aware chatbot for the AI Game."""
    
    def __init__(self):
        self.analyzer = pipeline("sentiment-analysis")
        
    def get_sentiment(self, text):
        """Analyze text sentiment."""
        result = self.analyzer(text)[0]
        return result["label"], result["score"]
    
    def respond(self, text):
        """Generate response based on sentiment."""
        sentiment, confidence = self.get_sentiment(text)
        
        if sentiment == "POSITIVE":
            responses = ["😊 Great!", "🎉 Awesome!", "✨ Fantastic!"]
        else:
            responses = ["🤔 Let\'s try again!", "💪 You can do it!", "🌟 Keep going!"]
        
        return random.choice(responses)
'''

with open("game_modules/chatbot.py", "w") as f:
    f.write(chatbot_code)

print("💾 Chatbot saved to game_modules/chatbot.py")

---

## 🔮 What's Coming Next!

In **Session 5: AI Game + Agents**, you'll:
- Combine everything into an interactive game
- Use your image generator, classifier, and chatbot
- Learn about AI agents (optional bonus)

---

## 🎯 Session 4 Wrap-Up

**You learned:**
- ✅ Sentiment analysis with Transformers
- ✅ Building a sentiment-aware chatbot
- ✅ Voice input with Whisper (optional)
- ✅ Voice output with TTS (optional)

🎉 **Your AI can now understand emotions and speak!**